In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ActorCriticNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_size=64):
        super(ActorCriticNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.actor_head = nn.Linear(hidden_size, action_size)
        self.critic_head = nn.Linear(hidden_size, 1)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        action_logits = self.actor_head(x)
        value = self.critic_head(x)
        return action_logits, value

import torch
import torch.optim as optim
import numpy as np
import random
from torch.distributions import Categorical
from IPython.display import clear_output
class ActorCriticAgent:
    def __init__(self, state_size, action_size, learning_rate=1e-3, gamma=0.99):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma
        self.best_reward = -float('inf')
        self.model = ActorCriticNetwork(state_size, action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def take_action(self, state, greedy=True):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            action_logits, _ = self.model(state)
            probs = torch.softmax(action_logits, dim=-1).cpu()
            dist = Categorical(probs)
            if greedy:
                # 訓練時使用隨機行為
                return dist.sample().item()
            else:
                # 測試時使用貪婪行為
                return probs.argmax(axis=-1).item()
    
    def actor_loss(self, log_probs, advantages):
        return -(log_probs * advantages).mean()

    def critic_loss(self, values, returns):
        return F.mse_loss(values, returns)

    def fit(self, env, num_episodes=1000, max_steps=200):
        for episode in range(num_episodes):
            state = env.reset()
            rewards = []
            log_probs = []
            values = []
            done = False
            step = 0
            while not done:
                action = self.take_action(state)
                next_state, reward, done, _ = env.step(action)
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
                action_logits, value = self.model(state_tensor)
                probs = torch.softmax(action_logits, dim=-1)

                dist = Categorical(probs)
                log_prob = dist.log_prob(torch.tensor(action).to(self.device))
            
                rewards.append(reward)
                log_probs.append(log_prob)
                values.append(value)
                state = next_state
                step += 1
            
            returns = []
            R = 0
            for r in reversed(rewards): # traverse reverse rewards, calculate returns
                R = r + self.gamma * R
                returns.insert(0, R) # insert at the beginning of the list
            returns = torch.tensor(returns).float().to(self.device)
            values = torch.cat(values).squeeze()
            advantages = (returns - values).detach() # Advantage estimation, using TD error

            log_probs = torch.stack(log_probs) # Convert list to tensor
            actor_loss = self.actor_loss(log_probs, advantages)
            critic_loss = self.critic_loss(values, returns)

            loss = actor_loss + critic_loss
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            reward_sum = sum(rewards)
            print(f"Episode: {episode + 1}, Loss: {loss.item():.4f}, Reward: {reward_sum:.2f}")
            
            if reward_sum > self.best_reward:
                self.best_reward = reward_sum
                self.save("best_actor_critic.pth")
                self.state_dict = self.model.state_dict() 
            print(f"新最佳獎勵: {self.best_reward}，已儲存模型")
                
            clear_output(wait=True)
        self.model.load_state_dict(self.state_dict)

    def save(self, filepath):
        torch.save(self.model.state_dict(), filepath)

    def load(self, filepath):
        self.model.load_state_dict(torch.load(filepath))
        self.model.eval()  # Set the model to evaluation mode after loading


In [2]:
# --- 設定 ---
ENV_ID = "CartPole-v1" # 環境 ID
GIF_FILENAME = "cartpole_random_episode.gif" # 輸出的 GIF 檔名
FRAME_DURATION_MS = 40 # 每幀顯示的毫秒數 (40ms = 25 FPS)
MAX_FRAMES = 500 # 最多儲存的幀數 (CartPole 通常一個 episode 不會太長)

# --- 創建環境 ---
# render_mode='rgb_array' 模式是儲存 GIF 必需的

import gym
import matplotlib.pyplot as plt
env = gym.make(ENV_ID, render_mode='rgb_array')

In [3]:
class Env:
    def __init__(self):
        self.env = gym.make("CartPole-v1", render_mode='rgb_array')
        self.state = self.env.reset()[0]
        self.done = False
    def step(self, action):
        self.state, reward, terminated, truncated, info = self.env.step(action)
        self.done = terminated or truncated
        return self.state, reward, self.done, info
    def reset(self):
        self.state = self.env.reset()[0]
        self.done = False
        return self.state
    def render(self):
        return self.env.render()
    def close(self):
        self.env.close()

In [4]:
agent = ActorCriticAgent(env.observation_space.shape[0], env.action_space.n, learning_rate=1e-3)
agent.fit(Env(), num_episodes=5000)

KeyboardInterrupt: 

In [ ]:
from PIL import Image
frames = []
ENV = Env()
done = False
state = ENV.reset()
actions = list()
while not done:
    action = agent.take_action(state, greedy=False)
    actions += [action]
    next_state, _, done, _ = ENV.step(action)
    state = next_state
    frame = ENV.render()
    frames.append(frame)
ENV.close()

image_objects = [Image.fromarray(np.uint8(frame)) for frame in frames]
image_objects[0].save(
                    GIF_FILENAME,
                    save_all=True,
                    append_images=image_objects[1:],
                    optimize=False, # 可以嘗試 optimize=True，但有時會導致顏色或透明度問題
                    duration=FRAME_DURATION_MS,
                    loop=0
                )